# PyTorch GPT Model Training Notebook

This notebook contains the complete implementation of a GPT-like model for pre-training, including all dependencies.

## Install Required Dependencies

First, let's install the required packages for the model training.

In [13]:
# # Install required packages
# !pip install torch torchvision torchaudio
# !pip install tokenizers
# !pip install tensorboard
# !pip install tqdm
# !pip install matplotlib
# !pip install numpy

## Utility Functions and Classes

Here we define all utility functions and classes that are used in the model.

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import tokenizers


class TextGenerator:
    def __init__(
        self,
        model: nn.Module,
        tokenizer: tokenizers.Tokenizer,
        device,
        padding_side="right",
    ) -> None:
        self.tokenizer = tokenizer
        self.device = device
        # self.padding_side = padding_side
        if isinstance(model, nn.DataParallel):
            print("该模型使用了DataParallel")
            self.model = model.module
        else:
            self.model = model
        self.seq_max_len = self.model.args.seq_max_len
        self.padding_side = padding_side
        self.tokenizer.enable_padding(direction=padding_side, length=self.seq_max_len)
        self.tokenizer.enable_truncation(
            max_length=self.seq_max_len, direction=padding_side
        )

    def generate(
        self,
        start_token: str,
        gen_seq_len=30,
        temperature=0.7,
        frequency_penalty=1.5,
        top_k=20,
        print_out=True,
    ):
        with torch.no_grad():
            self.model.eval()
            tokens = [start_token]  # 无padding
            # 初始化全序列ID列表（包含start_token）
            all_token_ids = self.tokenizer.encode(start_token).ids

            for i in range(gen_seq_len):

                all_token_ids = self.tokenizer.encode("".join(tokens)).ids
                # 模型前向传播
                input_tensor = (
                    torch.tensor(all_token_ids).int().unsqueeze(0).to(self.device)
                )
                out = self.model(input_tensor)

                # 根据padding方向选择logits位置
                if self.padding_side == "right":
                    logits = out[0, len(tokens) - 1, :]
                elif self.padding_side == "left":
                    logits = out[0, -1, :]
                else:
                    raise ValueError("padding_side must be 'right' or 'left'")

                # 频率惩罚（修复后）
                if frequency_penalty != 0:
                    # 使用当前全序列计算频率
                    tokens_tensor = torch.tensor(all_token_ids, device=self.device)
                    unique, counts = torch.unique(tokens_tensor, return_counts=True)

                    # 创建惩罚张量（与logits同设备）
                    penalty = torch.zeros_like(logits)
                    penalty[unique] = counts.float() * frequency_penalty
                    logits = logits - penalty

                # top_k 处理
                if top_k is not None and top_k > 0:
                    # 获取 top_k 以外的 token 索引
                    indices_to_remove = (
                        logits < torch.topk(logits, top_k)[0][..., -1, None]
                    )
                    logits[indices_to_remove] = -float("Inf")

                # 采样下一个token
                probabilities = F.softmax(logits / temperature, dim=-1)
                next_token_id = probabilities.multinomial(num_samples=1).item()

                # 更新序列
                tokens.append(
                    self.tokenizer.decode([next_token_id], skip_special_tokens=False)
                )
                if print_out:
                    print(tokens[-1], end=" ", flush=True)

            return tokens


class DebugTimer:
    def __init__(self, name=None):
        self.start_time = None
        self.name = name

    def __call__(self, func):
        def wrapper(*args, **kwargs):
            self.timer_start(self.name)
            result = func(*args, **kwargs)
            self.timer_stop()
            return result

        return wrapper

    def timer_start(self, name=None):
        self.start_time = time.perf_counter()
        if name is not None:
            self.name = name
        print(f"{self.name}:", end="")

    def timer_stop(self):
        elapsed_time = round(time.perf_counter() - self.start_time, 4)
        print(f"{elapsed_time}s")


def _format_string(s, length, fill_char=" "):
    """辅助函数：将字符串格式化为指定长度，不足部分用 fill_char 填充"""
    return s.ljust(length, fill_char)


def model_structure(model):
    """打印模型结构信息，包括权重名称、形状和参数数量"""
    print("-" * 90)
    print(
        "|"
        + _format_string("weight name", 31)
        + "|"
        + _format_string("weight shape", 42)
        + "|"
        + _format_string("number", 13)
        + "|"
    )
    print("-" * 90)

    total_params = 0
    type_size = 1  # 如果是浮点数就是4

    for key, param in model.named_parameters():
        # 格式化输出
        formatted_key = _format_string(key, 30)
        shape_str = _format_string(str(param.shape), 40)
        param_count = param.numel()
        formatted_count = _format_string(str(param_count), 10)

        print(f"| {formatted_key} | {shape_str} | {formatted_count} |")
        total_params += param_count

    print("-" * 90)
    print(f"The total number of parameters: {total_params}")
    print(
        f"The parameters of Model {model._get_name()}: {total_params * type_size / 1e6:.4f}M"
    )
    print("-" * 90)
    return total_params


class WarmUpCosineLR(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, total_steps, warmup_steps, min_lr=0, last_step=-1):
        """
        Args:
            optimizer (Optimizer): 包装的优化器。
            total_steps (int): 总的训练步数。
            warmup_steps (int): warm-up 的步数。
            min_lr (float): 最小学习率，默认为 0。
            last_epoch (int): 上一轮的索引，默认为 -1。
        """
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps
        self.min_lr = min_lr
        last_epoch = last_step
        super(WarmUpCosineLR, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        if self.last_epoch < self.warmup_steps:
            # Warm-up 阶段：线性增加学习率
            return [
                base_lr * (self.last_epoch + 1) / self.warmup_steps
                for base_lr in self.base_lrs
            ]
        else:
            # 余弦退火阶段
            current_step = self.last_epoch - self.warmup_steps
            total_cosine_steps = self.total_steps - self.warmup_steps
            if total_cosine_steps <= 0:
                return [base_lr for base_lr in self.base_lrs]
            return [
                self.min_lr
                + (base_lr - self.min_lr)
                * (
                    1
                    + torch.cos(
                        torch.tensor(current_step / total_cosine_steps * torch.pi)
                    )
                )
                / 2
                for base_lr in self.base_lrs
            ]


class WarmUpStableDecayLR(torch.optim.lr_scheduler._LRScheduler):
    def __init__(
        self,
        optimizer,
        total_steps,
        warmup_steps,
        stable_steps,
        decay_mode="linear",
        min_lr=0,
        last_step=-1,
    ):
        """
        WarmUpStableDecay学习率调度器
        该调度器包含三个阶段：
        1. 预热阶段：从min_lr线性增长至基础学习率
        2. 稳定阶段：保持基础学习率不变
        3. 衰减阶段：线性衰减至min_lr

        Args:
            optimizer (Optimizer): 包装的优化器
            total_steps (int): 总的训练步数
            warmup_steps (int): 预热步数
            stable_steps (int): 稳定步数
            decay_mode (str): 衰减模式，可选linear, exp
            min_lr (float): 最小学习率，默认为0
            last_epoch (int): 上一步的索引，默认为-1
        """
        assert decay_mode in ["linear", "exp"], "decay_mode 必须是 linear 或 exp"
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps
        self.stable_steps = stable_steps
        self.min_lr = min_lr
        self.decay_mode = decay_mode
        last_epoch = last_step
        super(WarmUpStableDecayLR, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        if self.last_epoch < self.warmup_steps:
            # 预热阶段：从min_lr线性增长至基础学习率
            return [
                self.min_lr
                + (base_lr - self.min_lr) * (self.last_epoch + 1) / self.warmup_steps
                for base_lr in self.base_lrs
            ]
        elif self.last_epoch < self.warmup_steps + self.stable_steps:
            # 稳定阶段：保持基础学习率不变
            return self.base_lrs
        else:
            # 衰减阶段：从基础学习率指数衰减至min_lr
            # 计算衰减步数
            decay_steps = self.last_epoch - (self.warmup_steps + self.stable_steps)
            # 总衰减步数
            total_decay_steps = self.total_steps - (
                self.warmup_steps + self.stable_steps
            )

            if total_decay_steps <= 0:
                # 如果没有衰减阶段，返回基础学习率
                return self.base_lrs

            # 计算每一步的衰减因子
            lrs = []
            for base_lr in self.base_lrs:
                if self.decay_mode == "linear":
                    # 线性衰减
                    progress = min(decay_steps / total_decay_steps, 1.0)
                    current_lr = base_lr + (self.min_lr - base_lr) * progress

                elif self.decay_mode == "exp":
                    # 指数衰减
                    if base_lr != 0 and self.min_lr >= 0 and base_lr > self.min_lr:
                        # 计算衰减率，确保在总衰减步数后达到min_lr
                        # lr = base_lr * decay_rate^(total_decay_steps) = min_lr
                        # 所以 decay_rate = (min_lr / base_lr)^(1 / total_decay_steps)
                        decay_rate = pow(
                            max(self.min_lr / base_lr, 1e-10), 1.0 / total_decay_steps
                        )
                        # 计算当前步骤的学习率
                        current_lr = base_lr * pow(decay_rate, decay_steps)
                    elif self.min_lr >= 0 and base_lr <= self.min_lr:
                        # 如果基础学习率已经小于等于最小学习率，则保持最小学习率
                        current_lr = self.min_lr
                lrs.append(current_lr)

            return lrs

## Dataset Class

Define the dataset class for streaming text data.

In [15]:
import torch
import collections
from tqdm import tqdm
import json
import sys
import random


class StreamingTextDataset(torch.utils.data.Dataset):
    """
    流式文本数据集，避免将整个数据集加载到内存中
    只存储文件路径和行位置信息，在需要时才读取特定行
    """

    def __init__(
        self,
        data_dir: str,
        tokenizer: tokenizers.Tokenizer,
        seq_max_len: int = 192,
        downsample: int = 1,
        batch: bool = None,  # 兼容性参数
        re_tokenize: bool = False,
        padding_side: str = "right",
    ):
        super().__init__()
        self.data_dir = data_dir
        self.tokenizer = tokenizer
        self.seq_max_len = seq_max_len
        self.re_tokenize = re_tokenize
        self.padding_side = padding_side

        # 构建行索引，只存储行的偏移位置而不是内容
        self.line_offsets = []
        self._build_line_index(downsample)

    def _build_line_index(self, downsample: int):
        """构建行偏移索引，避免加载整个文件"""
        with open(self.data_dir, "rb") as f:
            offset = 0
            line_count = 0
            while True:
                self.line_offsets.append(offset)
                line = f.readline()
                if not line:
                    break

                offset += len(line)
                line_count += 1
            self.line_offsets = random.sample(
                self.line_offsets, k=int(len(self.line_offsets) * downsample)
            )

    def pad_seq(
        self,
        seq: list[int],
        max_len: int,
        truncation=True,
        padding_value=0,
        padding_side="left",
    ):
        """
        对序列进行填充
        Args:
            seq: 序列
            max_len: 最大长度
            padding_value: 填充值
            padding_side: 填充方向
        Returns:
            填充后的序列
        """
        # 截断
        if truncation:
            if padding_side == "right":
                seq = seq[:max_len]
            elif padding_side == "left":
                seq = seq[-max_len:]
            else:
                raise ValueError("padding_side must be 'left' or 'right'")

        # 填充
        if len(seq) < max_len:
            if padding_side == "left":
                seq = [padding_value] * (max_len - len(seq)) + seq
            elif padding_side == "right":
                seq = seq + [padding_value] * (max_len - len(seq))
            else:
                raise ValueError("padding_side must be 'left' or 'right'")

        return seq

    def __len__(self):
        return len(self.line_offsets)

    def __getitem__(self, index):
        """
        根据索引获取数据样本，只在需要时读取特定行
        """
        # 根据索引定位并读取特定行
        with open(self.data_dir, "r", encoding="utf-8") as f:
            f.seek(self.line_offsets[index])
            line = f.readline().strip()

        # 进行tokenization
        if self.re_tokenize:
            # 如果需要重新分词，直接使用原始字符串
            raw = self.tokenizer.encode(line).ids
        else:
            # 如果使用预分词数据，需要先将字符串分割成列表
            raw = self.tokenizer.encode(line.split(" "), is_pretokenized=True).ids

        raw = self.pad_seq(
            raw,
            max_len=self.seq_max_len,
            truncation=True,
            padding_value=0,
            padding_side=self.padding_side,
        )

        # 将列表转换为tensor
        raw_tensor = torch.tensor(raw, dtype=torch.long)

        return (raw_tensor[:-1].contiguous(), raw_tensor[1:].contiguous())

## Model Architecture

Define the model architecture for the GPT-like model.

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint
from dataclasses import dataclass
import math


@dataclass
class MyLMArgs:
    d_model: int
    d_inner: int
    n_layers: int
    vocab_size: int
    seq_max_len: int
    use_moe: bool = False
    n_heads: int = None
    n_experts: int = 4
    n_experts_per_tok: int = 2
    d_conv: int = 3
    conv_bias: bool = True
    ffn_bias: bool = False
    attn_bias: bool = False
    d_head: int = 64
    dropout: float = 0.1
    base_init_std: float = 0.02  # 基础初始化标准差
    resid_pdrop: float = 0.1  # 残差连接dropout
    resid_scale: float = 1.0  # 残差流缩放参数
    layer_scale: float = 1.0  # 层缩放参数
    use_deepnet_scaling: bool = True  # 是否使用DeepNet缩放策略


class RMSNorm(torch.nn.Module):
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps
        self._reset_parameters()

    def _reset_parameters(self, base_init_std=0.02):
        # 初始化权重为1
        with torch.no_grad():
            self.weight.fill_(1.0)

    def forward(self, x):
        input_dtype = x.dtype
        x = x.to(torch.float32)
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * x.to(input_dtype)


class CausalConv1d(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride=1,
        dilation=1,
        groups=1,
        bias=True,
    ):
        super(CausalConv1d, self).__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size,
            stride=stride,
            padding=self.pad,
            dilation=dilation,
            groups=groups,
            bias=bias,
        )

    def forward(self, input):
        return self.conv(input)[:, :, : -self.pad]


class GPT2PositionEmbedding(nn.Module):
    def __init__(self, seq_max_len, d_model):
        super().__init__()
        self.pos_emb = nn.Embedding(seq_max_len, d_model)
        self._reset_parameters()

    def _reset_parameters(self, base_init_std=0.02):
        nn.init.normal_(self.pos_emb.weight, std=base_init_std)

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape

        assert (
            seq_len <= self.pos_emb.num_embeddings
        ), f"序列长度 {seq_len} 超过预设最大值 {self.pos_emb.num_embeddings}"  # 检查序列长度是否超限
        # 生成位置编码并相加
        pos = torch.arange(seq_len).to(x.device)  # (seq_len,)
        pos_emb = self.pos_emb(pos)  # (seq_len, d_model)
        pos_emb = pos_emb.unsqueeze(0)  # (1, seq_len, d_model)
        return x + pos_emb  # 广播到 (batch_size, seq_len, d_model)


class MyPositionEmbedding(nn.Module):
    def __init__(self, d_model, d_out, d_inner=None):
        super().__init__()
        if d_inner is None:
            d_inner = d_model // 16
        self.d_inner = d_inner
        self.d_model = d_model
        self.gru = nn.GRU(d_inner, d_inner, bias=False, batch_first=True)
        self.pos_conv = nn.Conv1d(d_model, d_inner, 1)
        # self.proj_conv = nn.AdaptiveAvgPool1d(d_model - d_inner)
        self.proj_conv = nn.Conv1d(d_model, d_model - d_inner, 1)
        self.linear = nn.Linear(d_model, d_out, bias=False)
        # self.up_proj = nn.Linear(d_inner, d_model, bias=False)

    def _reset_parameters(self, base_init_std=0.02):
        torch.nn.init.normal_(self.linear.weight, std=base_init_std)
        # 初始化其他层的权重
        if hasattr(self, "pos_conv") and self.pos_conv.weight is not None:
            torch.nn.init.normal_(self.pos_conv.weight, std=base_init_std)
        if hasattr(self, "proj_conv") and self.proj_conv.weight is not None:
            torch.nn.init.normal_(self.proj_conv.weight, std=base_init_std)
        # GRU层的初始化
        if hasattr(self, "gru"):
            for name, param in self.gru.named_parameters():
                if "weight" in name:
                    torch.nn.init.normal_(param, std=base_init_std)
                elif "bias" in name:
                    torch.nn.init.zeros_(param)

    def forward(self, x):
        res = x
        x = x.transpose(1, 2)
        x_proj = self.proj_conv(x).transpose(1, 2)  # 下采样非时间特征
        pos = self.pos_conv(x).transpose(1, 2)  # 下采样时间特征
        pos, _ = self.gru(pos)  # 时间特征RNN
        x = self.linear(torch.cat([pos, x_proj], dim=-1))  # 维度融合

        return x + res


class ALiBi(nn.Module):
    def __init__(self, num_heads):
        super().__init__()
        self.num_heads = num_heads
        slopes = torch.Tensor(self._get_slopes(num_heads))
        self.register_buffer("slopes", slopes)

    def _get_slopes(self, n):
        def get_slopes_power_of_2(n):
            start = 2 ** (-(2 ** -(math.log2(n) - 3)))
            ratio = start
            return [start * ratio**i for i in range(n)]

        if math.log2(n).is_integer():
            return get_slopes_power_of_2(n)
        else:
            closest_power_of_2 = 2 ** math.floor(math.log2(n))
            return (
                get_slopes_power_of_2(closest_power_of_2)
                + self._get_slopes(2 * closest_power_of_2)[0::2][
                    : n - closest_power_of_2
                ]
            )

    def forward(self, seq_len, batch_size, device):
        # 生成相对位置矩阵
        context_position = torch.arange(seq_len, device=device)[:, None]
        memory_position = torch.arange(seq_len, device=device)[None, :]
        relative_position = torch.abs(
            context_position - memory_position
        )  # (seq_len, seq_len)

        # 为每个头生成偏置矩阵 (num_heads, seq_len, seq_len)
        bias = relative_position[None, ...] * self.slopes[:, None, None]
        bias = -bias  # ALiBi 的负偏置

        # 扩展为 (batch_size * num_heads, seq_len, seq_len)
        bias = bias.repeat(batch_size, 1, 1)  # 直接复制到每个样本
        return bias


class Attention(nn.Module):
    """带有sigmoid激活的LLaMA 注意力机制"""

    def __init__(self, args: MyLMArgs, base_init_std=0.02):
        super().__init__()
        self.d_model = args.d_model
        self.n_heads = args.n_heads or (args.d_model // args.d_head)
        self.d_head = args.d_head
        self.seq_max_len = args.seq_max_len

        # 注意力投影层
        self.q_proj = nn.Linear(args.d_model, args.d_model)
        self.k_proj = nn.Linear(args.d_model, args.d_model)
        self.v_proj = nn.Linear(args.d_model, args.d_model)
        self.o_proj = nn.Linear(args.d_model, args.d_model)

        # RoPE位置编码缓存
        self.register_buffer(
            "cos_cached", torch.zeros(1, 1, args.seq_max_len, args.d_head)
        )
        self.register_buffer(
            "sin_cached", torch.zeros(1, 1, args.seq_max_len, args.d_head)
        )

        self.attn_dropout = nn.Dropout(args.dropout)
        self.resid_dropout = nn.Dropout(args.dropout)

        # 初始化RoPE
        self._init_rope()
        self._reset_parameters(base_init_std=base_init_std)

    def _reset_parameters(self, base_init_std=0.02):
        # 初始化线性层权重
        torch.nn.init.normal_(self.q_proj.weight, std=base_init_std)
        torch.nn.init.normal_(self.k_proj.weight, std=base_init_std)
        torch.nn.init.normal_(self.v_proj.weight, std=base_init_std)
        torch.nn.init.normal_(self.o_proj.weight, std=base_init_std)

    def _init_rope(self):
        """初始化RoPE位置编码"""
        d_head_half = self.d_head // 2
        # 创建频率数组，长度为d_head_half
        inv_freq = 1.0 / (
            10000 ** (torch.arange(0, d_head_half, dtype=torch.float) / d_head_half)
        )

        t = torch.arange(self.seq_max_len, dtype=torch.float)
        # 计算位置频率
        freqs = torch.einsum("i,j->ij", t, inv_freq)

        # 扩展到完整维度并添加批次和头数维度
        emb = torch.cat((freqs, freqs), dim=-1)  # (seq_len, d_head)
        # 使用register_buffer更新缓存，而不是直接赋值
        self.register_buffer(
            "cos_cached", emb.cos().unsqueeze(0).unsqueeze(0)
        )  # (1, 1, seq_len, d_head)
        self.register_buffer(
            "sin_cached", emb.sin().unsqueeze(0).unsqueeze(0)
        )  # (1, 1, seq_len, d_head)

    def _rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        """旋转一半维度"""
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    def _apply_rotary_pos_emb(
        self, q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor
    ):
        """应用RoPE位置编码"""
        # 调整cos和sin的维度以匹配q和k的序列长度
        cos = cos[:, :, : q.size(2), :]  # (1, 1, seq_len, d_head)
        sin = sin[:, :, : q.size(2), :]  # (1, 1, seq_len, d_head)

        # 将q和k分割为两半用于旋转操作
        q_embed = (q * cos) + (self._rotate_half(q) * sin)
        k_embed = (k * cos) + (self._rotate_half(k) * sin)
        return q_embed, k_embed

    def forward(self, x: torch.Tensor, token_ids=None) -> torch.Tensor:
        batch_size, seq_len, _ = x.size()

        # 计算QKV
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # 重塑为多头形式
        q = q.view(batch_size, seq_len, self.n_heads, self.d_head).transpose(
            1, 2
        )  # (batch, heads, seq, head_dim)
        k = k.view(batch_size, seq_len, self.n_heads, self.d_head).transpose(
            1, 2
        )  # (batch, heads, seq, head_dim)
        v = v.view(batch_size, seq_len, self.n_heads, self.d_head).transpose(
            1, 2
        )  # (batch, heads, seq, head_dim)

        # 应用RoPE位置编码
        cos = self.cos_cached[:, :, :seq_len, :]  # (1, 1, seq_len, d_head)
        sin = self.sin_cached[:, :, :seq_len, :]  # (1, 1, seq_len, d_head)
        q, k = self._apply_rotary_pos_emb(q, k, cos, sin)

        # 计算注意力分数
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.d_head))
        causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=att.device)).view(
            1, 1, seq_len, seq_len
        )
        att = att.masked_fill(causal_mask == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        # 应用注意力权重
        y = att @ F.sigmoid(v)  # (batch, heads, seq, head_dim)
        y = (
            y.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        )  # 重新组合多头
        # 输出投影
        y = self.resid_dropout(self.o_proj(y))
        return y


class GatedAttention(nn.Module):
    """Qwen 门控注意力机制"""

    def __init__(self, args: MyLMArgs, base_init_std=0.02):
        super().__init__()
        self.d_model = args.d_model
        self.n_heads = args.n_heads or (args.d_model // args.d_head)
        self.d_head = args.d_head
        self.seq_max_len = args.seq_max_len

        # 注意力投影层
        self.q_proj = nn.Linear(args.d_model, args.d_model, bias=args.attn_bias)
        self.k_proj = nn.Linear(args.d_model, args.d_model, bias=args.attn_bias)
        self.v_proj = nn.Linear(args.d_model, args.d_model, bias=args.attn_bias)
        self.o_proj = nn.Linear(args.d_model, args.d_model, bias=args.attn_bias)
        self.gate = nn.Linear(args.d_model, args.d_model, bias=False)

        # RoPE位置编码缓存
        self.register_buffer(
            "cos_cached", torch.zeros(1, 1, args.seq_max_len, args.d_head)
        )
        self.register_buffer(
            "sin_cached", torch.zeros(1, 1, args.seq_max_len, args.d_head)
        )

        self.attn_dropout = nn.Dropout(args.dropout)
        self.resid_dropout = nn.Dropout(args.dropout)

        # 初始化RoPE
        self._init_rope()
        self._reset_parameters(base_init_std=base_init_std)

    def _reset_parameters(self, base_init_std=0.02):
        # 初始化线性层权重
        torch.nn.init.normal_(self.q_proj.weight, std=base_init_std)
        torch.nn.init.normal_(self.k_proj.weight, std=base_init_std)
        torch.nn.init.normal_(self.v_proj.weight, std=base_init_std)
        torch.nn.init.normal_(self.o_proj.weight, std=base_init_std)
        torch.nn.init.normal_(self.gate.weight, std=base_init_std)

    def _init_rope(self):
        """初始化RoPE位置编码"""
        d_head_half = self.d_head // 2
        # 创建频率数组，长度为d_head_half
        inv_freq = 1.0 / (
            10000 ** (torch.arange(0, d_head_half, dtype=torch.float) / d_head_half)
        )

        t = torch.arange(self.seq_max_len, dtype=torch.float)
        # 计算位置频率
        freqs = torch.einsum("i,j->ij", t, inv_freq)

        # 扩展到完整维度并添加批次和头数维度
        emb = torch.cat((freqs, freqs), dim=-1)  # (seq_len, d_head)
        # 使用register_buffer更新缓存，而不是直接赋值
        self.register_buffer(
            "cos_cached", emb.cos().unsqueeze(0).unsqueeze(0)
        )  # (1, 1, seq_len, d_head)
        self.register_buffer(
            "sin_cached", emb.sin().unsqueeze(0).unsqueeze(0)
        )  # (1, 1, seq_len, d_head)

    def _rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        """旋转一半维度"""
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    def _apply_rotary_pos_emb(
        self, q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor
    ):
        """应用RoPE位置编码"""
        # 调整cos和sin的维度以匹配q和k的序列长度
        cos = cos[:, :, : q.size(2), :]  # (1, 1, seq_len, d_head)
        sin = sin[:, :, : q.size(2), :]  # (1, 1, seq_len, d_head)

        # 将q和k分割为两半用于旋转操作
        q_embed = (q * cos) + (self._rotate_half(q) * sin)
        k_embed = (k * cos) + (self._rotate_half(k) * sin)
        return q_embed, k_embed

    def forward(self, x: torch.Tensor, token_ids=None) -> torch.Tensor:
        batch_size, seq_len, _ = x.size()

        # 计算QKV
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        gate = F.sigmoid(self.gate(x))

        # 重塑为多头形式
        q = q.view(batch_size, seq_len, self.n_heads, self.d_head).transpose(
            1, 2
        )  # (batch, heads, seq, head_dim)
        k = k.view(batch_size, seq_len, self.n_heads, self.d_head).transpose(
            1, 2
        )  # (batch, heads, seq, head_dim)
        v = v.view(batch_size, seq_len, self.n_heads, self.d_head).transpose(
            1, 2
        )  # (batch, heads, seq, head_dim)

        # 应用RoPE位置编码
        cos = self.cos_cached[:, :, :seq_len, :]  # (1, 1, seq_len, d_head)
        sin = self.sin_cached[:, :, :seq_len, :]  # (1, 1, seq_len, d_head)
        q, k = self._apply_rotary_pos_emb(q, k, cos, sin)

        # 计算注意力分数
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.d_head))
        causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=att.device)).view(
            1, 1, seq_len, seq_len
        )
        att = att.masked_fill(causal_mask == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        # 应用注意力权重
        y = att @ v  # (batch, heads, seq, head_dim)
        y = (
            y.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        )  # 重新组合多头
        y = y * gate
        # 输出投影
        y = self.resid_dropout(self.o_proj(y))
        return y

class FFN(nn.Module):
    """LLaMA MLP层"""

    def __init__(self, args: MyLMArgs, base_init_std=0.02):
        super().__init__()
        self.args = args
        self.gate_proj = nn.Linear(args.d_model, args.d_inner, bias=False)
        self.up_proj = nn.Linear(args.d_model, args.d_inner, bias=False)
        self.down_proj = nn.Linear(args.d_inner, args.d_model, bias=False)
        self._reset_parameters(base_init_std=base_init_std)

    def _reset_parameters(self, base_init_std=0.02):
        print("on call _reset_parameters")
        # 初始化线性层权重
        torch.nn.init.normal_(self.gate_proj.weight, std=base_init_std)
        torch.nn.init.normal_(self.up_proj.weight, std=base_init_std)
        torch.nn.init.normal_(self.down_proj.weight, std=base_init_std)

    def forward(self, x: torch.Tensor, token_ids=None) -> torch.Tensor:
        y = self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

        return y


class MoEFFN(nn.Module):
    def __init__(self, args: MyLMArgs):
        """
        激活参数量为 n_experts_per_token * ffn
        总参数量为 n_experts * ffn
        """
        super().__init__()
        self.args = args
        self.router = nn.Linear(args.d_model, args.n_experts)
        self.experts = nn.ModuleList([FFN(args) for _ in range(args.n_experts)])

    def _reset_parameters(self, base_init_std=0.02):
        # 路由器使用较小的标准差，避免初始化时偏向特定专家
        torch.nn.init.normal_(self.router.weight, std=base_init_std * 0.1)
        if self.router.bias is not None:
            nn.init.zeros_(self.router.bias)
        # 初始化所有专家
        for expert in self.experts:
            expert._reset_parameters(base_init_std)

    def forward(self, x, token_ids=None):
        # 路由块
        probs = F.softmax(self.router(x), dim=-1)
        top_k_probs, top_k_indices = torch.topk(
            probs, self.args.n_experts_per_tok, dim=-1
        )
        # 归一化概率
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        # 专家块
        expert_inputs = x.view(
            -1, x.shape[-1]
        )  # 展平出所有token [batch_size*seq_len, hidden_size]
        expert_inputs = expert_inputs.repeat_interleave(
            self.args.n_experts_per_tok, dim=0
        )
        flat_top_k_idx = top_k_indices.view(-1)  # 展平出所有token对应的专家索引
        expert_outputs = torch.zeros_like(expert_inputs)  # 分配专家输出

        # 遍历所有专家 处理对应token
        for expert_idx, expert in enumerate(self.experts):
            mask = flat_top_k_idx == expert_idx  # 该专家
            if mask.any():
                expert_outputs[mask] = expert(expert_inputs[mask], token_ids=token_ids)
        expert_outputs = (
            expert_outputs.view(*top_k_probs.shape, -1) * top_k_probs.unsqueeze(-1)
        ).sum(
            dim=2
        )  # 乘以对应权重求和得到最终输出

        return expert_outputs


class MyLMDecoderLayer(nn.Module):
    """
    混合Transformer块
    门控注意力 : v激活注意力 = 1:1
    """

    def __init__(self, args: MyLMArgs, layer_idx=0, base_init_std=0.02):
        super().__init__()
        self.args = args
        self.attn = (
            GatedAttention(args, base_init_std=base_init_std)
            if layer_idx % 2 == 0
            else Attention(args, base_init_std=base_init_std)
        )
        self.mlp = (
            MoEFFN(args) if args.use_moe else FFN(args, base_init_std=base_init_std)
        )
        self.input_layernorm = RMSNorm(args.d_model)
        self.post_attention_layernorm = RMSNorm(args.d_model)

    def forward(self, x: torch.Tensor, token_ids=None) -> torch.Tensor:
        # 注意力部分
        residual = x
        x = self.input_layernorm(x)
        x = self.attn(x, token_ids=token_ids)
        x = residual + x

        # MLP部分
        residual = x
        x = self.post_attention_layernorm(x)
        x = self.mlp(x, token_ids=token_ids)
        x = residual + x

        return x


class MyLM(nn.Module):
    """
    简化版LLaMA架构实现
    兼容MyLMArgs配置参数
    """

    def __init__(self, args: MyLMArgs):
        super().__init__()
        self.args = args
        base_init_std = args.base_init_std
        # 词嵌入层
        self.token_embedding = nn.Embedding(args.vocab_size, args.d_model)

        # Transformer块
        self.blocks = nn.ModuleList(
            [
                MyLMDecoderLayer(args, layer_idx, base_init_std=base_init_std)
                for layer_idx in range(args.n_layers)
            ]
        )

        # 输出层
        self.norm = RMSNorm(args.d_model)
        self.head = nn.Linear(args.d_model, args.vocab_size, bias=False)
        self._reset_parameters(base_init_std=1 / math.sqrt(args.d_model))

    def _reset_parameters(self, base_init_std=0.02):
        torch.nn.init.normal_(self.token_embedding.weight, std=base_init_std)
        # for module in self.modules():
        # if isinstance(module, nn.Linear):
        #     if module.bias is not None:
        #         module.bias.data.zero_()
        # if isinstance(module, nn.Embedding):
        #     torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x: torch.Tensor, token_ids=None) -> torch.Tensor:
        """
        前向传播

        Args:
            x: 输入张量，形状为(batch_size, seq_len)
            token_ids: token ID，用于某些特殊操作

        Returns:
            输出张量，形状为(batch_size, seq_len, vocab_size)
        """
        # 词嵌入
        x = self.token_embedding(x)  # (batch_size, seq_len, d_model)

        # 通过Transformer块
        for block in self.blocks:
            x = block(x, token_ids=token_ids)

        # 最终归一化和输出投影
        x = self.norm(x)
        logits = self.head(x)

        return logits

## Pre-training Configuration and Trainer

Define the training configuration and trainer class.

In [17]:
import os

# 设置环境变量以解决OpenMP库重复初始化问题
os.environ["KMP_DUPLICATE_LIB_OK"] = "True"

import torch
import torch.nn as nn
import torch.utils.data
import torch.nn.functional as F
import torch

# import bitsandbytes as bnb
import numpy as np
import matplotlib.pyplot as plt
import tokenizers
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

import random
import gc
import json
import time

from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any


# ---------------------------------------------------#
#   工具组件
# ---------------------------------------------------#
t = DebugTimer()


@dataclass
class TrainingConfig:
    """训练配置参数"""

    # 数据配置
    data_dir: str = r"mini_data200v2.txt"
    tokenizer_dir: str = r"bpe_tokenizer_6k_0724_ChatML.json"
    model_save_dir: str = r"model\model_state.pth"
    ckpt_save_dir: str = r"ckpt\ckpt.pth"
    config_save_dir: str = r"config.json"
    # log_dir: str = r"logs/" + time.strftime("%Y%m%d-%H%M%S")
    log_dir: str = r"logs/20260102-143753"
    padding_side = "left"

    # 训练参数
    seed: int = 42
    epochs: int = 1
    batch_size: int = 16
    batch_acceleration: int = 3
    dataset_downsample: int = 0.005
    valset_rate: float = 0.005
    val_interval_step: int = 800
    seq_max_len = 200

    # 优化参数
    learning_rate: float = 4e-3
    min_learning_rate: float = 5e-5  # WSD LRS衰减到1%
    lr_decay_start_rate: int = 0.75  # 最后25%衰减
    warmup_steps: int = 75
    use_amp: bool = False

    model_args = MyLMArgs(
        d_model=384,
        d_inner=int(((384 * (6 / 3)) // 64) * 64),
        d_head=96,
        n_heads=None,
        n_layers=3,
        vocab_size=None,
        seq_max_len=seq_max_len,
        use_moe=False,
        n_experts=None,
        n_experts_per_tok=None,
        d_conv=None,
        conv_bias=None,
        ffn_bias=False,
        attn_bias=True,
        dropout=0.1,
    )

    # 新增参数：checkpoint保存间隔步数
    ckpt_interval_step: int = 1000
    # 新增参数：断点续训的checkpoint路径
    # resume_from: Optional[str] = r"ckpt\ckpt_epoch_0_step_6000.pth"
    resume_from: Optional[str] = None


class PreTrainer:
    def __init__(self, config: TrainingConfig):
        self.config = config
        self._set_seed()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = tokenizers.Tokenizer.from_file(config.tokenizer_dir)
        self.config.model_args.vocab_size = len(self.tokenizer.get_vocab())
        self.train_loader, self.val_loader = self._build_dataloader()
        self.model = self._build_model().to(self.device)
        self.criterion = nn.CrossEntropyLoss(ignore_index=0)
        self.optimizers, self.schedulers = self._build_optimizer()
        self.scaler = torch.GradScaler(self.device, enabled=config.use_amp)
        self.generator = TextGenerator(
            self.model, self.tokenizer, self.device, padding_side=config.padding_side
        )

        # 用于扩展的属性

        self.current_epoch = 0
        self.global_step = 0
        self.start_epoch = 0
        self.current_step = 0
        self.start_step = 0
        self.train_loss_log = []  # 将改为存储(step, loss)格式
        self.val_loss_log = []
        self.lr_log = []

        # 如果指定了resume_from路径，加载checkpoint
        if config.resume_from is not None:
            self.load_checkpoint(config.resume_from)

    def _set_seed(self):
        """设置随机种子"""
        random.seed(self.config.seed)
        np.random.seed(self.config.seed)
        torch.manual_seed(self.config.seed)
        torch.cuda.manual_seed(self.config.seed)
        torch.cuda.manual_seed_all(self.config.seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

    def _build_model(self):
        """构建模型"""
        model = MyLM(self.config.model_args)
        return model

    def _build_dataloader(self):
        """构建数据加载器"""
        dataset = StreamingTextDataset(
            self.config.data_dir,
            downsample=self.config.dataset_downsample,
            seq_max_len=self.config.seq_max_len,
            tokenizer=self.tokenizer,
            re_tokenize=False,
            batch=False,
            padding_side=self.config.padding_side,
        )
        val_dataset_len = int(len(dataset) * self.config.valset_rate)
        train_dataset_len = len(dataset) - val_dataset_len
        train_dataset, val_dataset = torch.utils.data.random_split(
            dataset, [train_dataset_len, val_dataset_len]
        )

        train_loader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=self.config.batch_size,
            shuffle=True,
            pin_memory=True,
            num_workers=4,
        )
        val_loader = torch.utils.data.DataLoader(
            val_dataset,
            batch_size=self.config.batch_size,
            shuffle=True,
            pin_memory=True,
            num_workers=1,
        )

        return train_loader, val_loader

    def _build_optimizer(self):
        """构建优化器"""
        muon_params = []
        other_params = []

        # 遍历所有命名参数
        for name, param in self.model.named_parameters():
            # 跳过不需要优化的参数
            if not param.requires_grad:
                continue

            # 根据参数维度和层类型判断
            if len(param.shape) == 2:
                # 进一步过滤：排除 Embedding 层和 LM Head 等
                if "embedding" in name.lower() or "embed" in name.lower():
                    other_params.append(param)
                elif (
                    "head" in name.lower()
                    or "classifier" in name.lower()
                    or "lm_head" in name.lower()
                ):
                    other_params.append(param)
                else:
                    # 检查是否为线性层权重（通常 bias 是 1D，权重是 2D）
                    if "weight" in name and "bias" not in name:
                        muon_params.append(param)
                    else:
                        other_params.append(param)
                # muon_params.append(param)
            else:
                # 1D、3D 及更高维参数都不适合 Muon
                other_params.append(param)

        # Muon With Aux AdamW
        optimizers = [
            torch.optim.Muon(
                muon_params,
                lr=self.config.learning_rate,
                adjust_lr_fn="match_rms_adamw",
                weight_decay=0.005,
            ),
            torch.optim.AdamW(
                other_params,
                lr=self.config.learning_rate,
                amsgrad=False,
                betas=(0.85, 0.999),
                eps=1e-6,
                weight_decay=0.005,
            ),
        ]

        # Full AdamW
        # optimizers = [
        # torch.optim.AdamW(
        #     self.model.parameters(),
        #     lr=self.config.learning_rate,
        #     amsgrad=False,
        # )
        # ]

        total_steps = (
            self.config.epochs
            * (len(self.train_loader) // self.config.batch_acceleration + 1)
        ) + 1

        # WarmUpCosineLR
        # schedulers = [
        #     WarmUpCosineLR(
        #         optimizer,
        #         total_steps=(
        #     self.config.epochs
        #     * (len(self.train_loader) // self.config.batch_acceleration + 1)
        # ) + 1,
        #         warmup_steps=self.config.warmup_steps,
        #         min_lr=self.config.min_learning_rate,
        #     ) for optimizer in optimizers]

        # WSD学习率调度器
        schedulers = [
            WarmUpStableDecayLR(
                optimizer,
                total_steps=total_steps,
                warmup_steps=self.config.warmup_steps,
                stable_steps=int(
                    self.config.lr_decay_start_rate * total_steps
                    - self.config.warmup_steps
                ),
                min_lr=self.config.min_learning_rate,
                decay_mode="exp",
            )
            for optimizer in optimizers
        ]

        return optimizers, schedulers

    def save_checkpoint(self, path: str, is_final: bool = False):
        # 检测是否是DataParallel模式
        if isinstance(self.model, nn.DataParallel):
            model_state_dict = self.model.module.state_dict()
        else:
            model_state_dict = self.model.state_dict()
        """保存checkpoint"""
        # 为多个优化器分别保存状态
        optimizer_states = []
        scheduler_states = []
        for opt, sched in zip(self.optimizers, self.schedulers):
            optimizer_states.append(opt.state_dict())
            scheduler_states.append(sched.state_dict())

        state = {
            "epoch": self.current_epoch,
            "global_step": self.global_step,
            "current_step": self.current_step,
            "model_state_dict": model_state_dict,
            "optimizer_states": optimizer_states,
            "scheduler_states": scheduler_states,
            "train_loss": self.train_loss_log[-1] if self.train_loss_log else None,
            "rng_states": {
                "torch": torch.get_rng_state(),
                "cuda": (
                    torch.cuda.get_rng_state_all()
                    if torch.cuda.is_available()
                    else None
                ),
                "random": random.getstate(),
                "numpy": np.random.get_state(),
            },
        }
        torch.save(state, path)
        if is_final:
            torch.save(model_state_dict, self.config.model_save_dir)

    def load_checkpoint(self, checkpoint_path: str):
        """加载checkpoint"""
        print(f"加载checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, weights_only=False)

        # 恢复模型状态
        self.model.load_state_dict(checkpoint["model_state_dict"])

        # 加载多个优化器的状态
        if "optimizer_states" in checkpoint and checkpoint["optimizer_states"]:
            for i, opt_state in enumerate(checkpoint["optimizer_states"]):
                if i < len(self.optimizers):
                    self.optimizers[i].load_state_dict(opt_state)
        else:
            # 兼容旧版本checkpoint
            self.optimizers.load_state_dict(checkpoint["optimizer_state_dict"])

        # 加载多个调度器的状态
        if "scheduler_states" in checkpoint and checkpoint["scheduler_states"]:
            for i, sched_state in enumerate(checkpoint["scheduler_states"]):
                if i < len(self.schedulers):
                    self.schedulers[i].load_state_dict(sched_state)
        else:
            # 兼容旧版本checkpoint
            self.schedulers.load_state_dict(checkpoint["scheduler_state_dict"])

        # 恢复训练状态
        self.current_epoch = checkpoint["epoch"]
        self.global_step = checkpoint["global_step"]
        self.start_epoch = checkpoint["epoch"]
        self.start_step = checkpoint["current_step"]

        # 恢复随机状态（防止数据shuffle混乱）
        rng_states = checkpoint["rng_states"]
        torch.set_rng_state(rng_states["torch"])
        if rng_states["cuda"] and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(rng_states["cuda"])
        random.setstate(rng_states["random"])
        np.random.set_state(rng_states["numpy"])

    def _train_step(self, inputs, targets):
        """单步训练（含梯度累加）"""
        self.model.train()
        inputs = inputs.to(self.device)
        targets = targets.to(self.device)

        with torch.autocast(str(self.device), enabled=self.config.use_amp):
            output = self.model(inputs)
            loss = self.criterion(
                output.view(-1, self.config.model_args.vocab_size), targets.view(-1)
            )

        loss = loss / self.config.batch_acceleration

        self.scaler.scale(loss).backward()
        # 梯度裁剪（可选）

        if ((self.current_step + 1) % self.config.batch_acceleration == 0) or (
            self.current_step + 1 == len(self.train_loader)
        ):
            # 对每个优化器进行梯度缩放和更新
            for optimizer in self.optimizers:
                self.scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            # 对每个优化器单独执行step
            for optimizer in self.optimizers:
                self.scaler.step(optimizer)
            self.scaler.update()
            # 对每个优化器单独清零梯度
            for optimizer in self.optimizers:
                optimizer.zero_grad(set_to_none=True)
            # 对每个调度器单独执行step
            for scheduler in self.schedulers:
                scheduler.step()

        return loss.item() * self.config.batch_acceleration

    def log(self):
        total_params = model_structure(self.model)
        print(f"本次训练参数：")
        print(f"词数: {self.tokenizer.get_vocab_size()}")
        val_dataset_len, train_dataset_len = len(self.val_loader.dataset), len(
            self.train_loader.dataset
        )
        print(f"上下文长度：{self.config.model_args.seq_max_len}")
        print(f"数据集数量：{val_dataset_len+train_dataset_len}")
        print(f"训练集数量：{train_dataset_len}")
        print(f"测试集数量：{val_dataset_len}")
        nums_token = self.config.model_args.seq_max_len * train_dataset_len
        print(f"Token数约：{nums_token/1e6:.3f}M")
        print(f"模型参数：{total_params/1e6:.3f}M")
        print(
            f"计算量：{(nums_token * total_params * 6)/1e12:.2f}TFLOPs * {self.config.epochs} = {(nums_token * total_params * 6 * self.config.epochs)/1e12:.2f}TFLOPs"
        )

    def train(self):
        """训练主循环"""
        gc.collect()
        writer = SummaryWriter(log_dir=self.config.log_dir)
        print("~~~训练咯~~~")

        # 多卡训练
        if torch.cuda.device_count() > 1:
            print(f"多卡训练: {torch.cuda.device_count()} 张GPU")
            self.model = nn.DataParallel(self.model)
        val_loss = 0
        for epoch in range(self.config.epochs):
            bar = tqdm(self.train_loader, unit="step")
            # 跳过已训练的epoch
            if epoch < self.start_epoch:
                print(f"跳过已训练的epoch: {epoch}")
                continue
            elif epoch == self.start_epoch:
                bar.update(self.start_step)

            self.current_epoch = epoch
            train_loss_sum = 0

            torch.cuda.empty_cache()

            for i, (train_inputs, train_targets) in enumerate(self.train_loader):
                # 跳过已训练的step
                if i <= self.start_step and epoch == self.start_epoch:
                    continue
                # # 如果是从断点恢复训练 打印提示
                # if i == self.start_step + 1:
                #     print(f"\r从断点恢复训练: {i}", end="")
                self.current_step = i
                loss = self._train_step(train_inputs, train_targets)

                train_loss_sum += loss
                self.train_loss_log.append((self.global_step, loss))
                # 添加TensorBoard训练损失记录
                writer.add_scalar("Loss/train", loss, self.global_step)
                # 记录学习率时添加step信息
                # 由于有多个优化器，我们记录第一个优化器的学习率
                self.lr_log.append(
                    (self.global_step, float(self.schedulers[0].get_last_lr()[0]))
                )
                self.global_step += 1
                # 添加TensorBoard学习率记录
                writer.add_scalar(
                    "LearningRate",
                    float(self.schedulers[0].get_last_lr()[0]),
                    self.global_step,
                )

                # 验证阶段
                if i % self.config.val_interval_step == 0:
                    val_loss = self.validate()
                    # 文本生成测试
                    test_text = self.generate_test("人工智能是")
                    writer.add_text(
                        "GeneratedText",
                        f"epoch_{epoch}_step{i}: {test_text}",
                        self.global_step,
                    )
                    self.val_loss_log.append((self.global_step, val_loss))
                    # 添加TensorBoard验证损失记录
                    writer.add_scalar("Loss/val", val_loss, self.global_step)

                # 进度显示
                bar.update(1)
                bar.postfix = f"train_loss: {loss:.2f} test_loss: {val_loss:.2f} lr: {self.schedulers[0].get_last_lr()[0]:.2e}"

                # 每n步直接保存checkpoint
                if self.global_step % self.config.ckpt_interval_step == 0:
                    val_loss = self.validate()
                    self.val_loss_log.append((self.global_step, val_loss))
                    # 生成带步数的checkpoint路径
                    ckpt_path = f"{self.config.ckpt_save_dir.rsplit('.', 1)[0]}_epoch_{self.current_epoch}_step_{self.global_step}.pth"
                    self.save_checkpoint(ckpt_path, is_final=False)

            bar.close()

            # 周期性验证
            val_loss = self.validate()
            self.val_loss_log.append((self.global_step, val_loss))

            # 文本生成测试

            # 修改：epoch级保存使用更明确的文件名格式
            self.save_checkpoint(
                f"{self.config.ckpt_save_dir.rsplit('.', 1)[0]}_epoch_{epoch}.pth",
                is_final=True,
            )

            # 每个epoch记录生成文本
            test_text = self.generate_test(gen_len=100)
            writer.add_text(
                "GeneratedText", f"epoch_{epoch}: {test_text}", self.global_step
            )
            print(f"学习率{self.schedulers[0].get_last_lr()}")

            print(
                f"Epoch: {epoch+1}/{self.config.epochs}, avg_train_loss: {train_loss_sum/len(self.train_loader)}, avg_test_loss: {val_loss}"
            )

        # 保存最终模型
        self.save_checkpoint(self.config.model_save_dir, is_final=True)

    def validate(self) -> float:
        """验证过程"""
        self.model.eval()
        val_loss_sum = 0
        with torch.no_grad():
            for val_inputs, val_targets in self.val_loader:
                val_inputs = val_inputs.to(self.device)
                val_targets = val_targets.to(self.device)
                with torch.autocast(str(self.device), enabled=self.config.use_amp):
                    val_output = self.model(val_inputs)
                    loss = self.criterion(
                        val_output.view(-1, self.config.model_args.vocab_size),
                        val_targets.view(-1),
                    )
                val_loss_sum += loss.item()
        return val_loss_sum / len(self.val_loader)

    def generate_test(self, start: str = "我", gen_len: int = 25):
        """文本生成测试（修改为返回文本）"""
        self.model.eval()
        ans = self.generator.generate(
            start_token=start, gen_seq_len=gen_len, print_out=False
        )
        ans = ans[len(start) :]  # 截掉start_token
        result = "".join(ans)
        # 保持原有控制台输出
        print(f"(input){start}-> {result}")
        return result

    def plot_losses(self):
        """绘制损失曲线"""
        fig, ax1 = plt.subplots(figsize=(16, 10))

        # 提取训练步骤和损失
        train_steps = [step for step, loss in self.train_loss_log]
        train_losses = [loss for step, loss in self.train_loss_log]

        # 提取验证步骤和损失
        val_steps = [step for step, loss in self.val_loss_log]
        val_losses = [loss for step, loss in self.val_loss_log]

        # 绘制左侧的训练损失和验证损失
        ax1.plot(train_steps, train_losses, label="Train Loss")  # 修改为使用step作为x轴
        ax1.plot(val_steps, val_losses, "o-", label="Test Loss")
        ax1.set_xlabel("Steps")
        ax1.set_ylabel("Loss")
        ax1.legend(loc="upper left")

        # 创建右侧y轴
        ax2 = ax1.twinx()
        ax2.plot(
            [step for step, _ in self.lr_log],  # 使用学习率日志中的step作为x轴
            [float(value) for _, value in self.lr_log],  # 使用value作为y轴
            label="Learning Rate",
            color="c",
            linestyle="--",
        )
        ax2.set_ylabel("Learning Rate")
        ax2.tick_params(axis="y")
        ax2.legend(loc="upper right")

        plt.title("Train and Test Loss Curves with Learning Rate")
        plt.show()

## Main Training Execution

Execute the training process.

In [ ]:
if __name__ == "__main__":
    config = TrainingConfig()
    config_dict = asdict(config.model_args)
    with open(config.config_save_dir, "w") as f:
        json.dump(config_dict, f, indent=4)
    trainer = PreTrainer(config)
    trainer.log()
    trainer.train()
    trainer.plot_losses()

    # 交互式测试
    MAX_LEN = 100
    T = 0.8
    while True:
        start = input("In>>")
        if start[:2] == "T=":
            T = float(start[2:])
            print(f"T={T}")
        else:
            print(
                f"T={T}\n"
                + "".join(
                    trainer.generator.generate(
                        start_token=start,
                        gen_seq_len=MAX_LEN,
                        temperature=T,
                        frequency_penalty=1.5,
                        print_out=False,
                    )
                )
            )

on call _reset_parameters
on call _reset_parameters
on call _reset_parameters
------------------------------------------------------------------------------------------
|weight name                    |weight shape                              |number       |
------------------------------------------------------------------------------------------
| token_embedding.weight         | torch.Size([6140, 384])                  | 2357760    |
| blocks.0.attn.q_proj.weight    | torch.Size([384, 384])                   | 147456     |
| blocks.0.attn.q_proj.bias      | torch.Size([384])                        | 384        |
| blocks.0.attn.k_proj.weight    | torch.Size([384, 384])                   | 147456     |
| blocks.0.attn.k_proj.bias      | torch.Size([384])                        | 384        |
| blocks.0.attn.v_proj.weight    | torch.Size([384, 384])                   | 147456     |
| blocks.0.attn.v_proj.bias      | torch.Size([384])                        | 384        |
| blocks.0.a

  0%|          | 0/207 [00:00<?, ?step/s]